# 04 — Greeks and Local Risk

I compare the analytical Greeks with independent finite differences before using them to explain local option P&L.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


In [2]:
from options_lab.black_scholes import black_scholes_price
from options_lab.greeks import black_scholes_greeks, to_market_greeks
from options_lab.numerical_greeks import numerical_greeks
from options_lab.risk import approximate_option_pnl

analytical=black_scholes_greeks(100.,100.,.05,1.,.20,"call")
numerical=numerical_greeks(100.,100.,.05,1.,.20,"call")
pd.DataFrame({"analytical":analytical.__dict__,"numerical":numerical.__dict__})

,analytical,numerical
delta,0.636831,0.636831
gamma,0.018762,0.018762
vega,37.524035,37.524034
theta,-6.414028,-6.414028
rho,53.232482,53.232482


In [3]:
spots=np.linspace(60,140,401)
g=black_scholes_greeks(spots,100.,.05,1.,.20,"call")
plt.figure(figsize=(8,5))
plt.plot(spots,g.gamma)
plt.xlabel("Current underlying price")
plt.ylabel("Gamma")
plt.title("Gamma versus Spot")
plt.show()

/tmp/ipykernel_6777/3808997337.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
base=black_scholes_price(100.,100.,.05,1.,.20,"call")
g=black_scholes_greeks(100.,100.,.05,1.,.20,"call")
rows=[]
for move in [-10,-5,-1,1,5,10]:
    exact=black_scholes_price(100+move,100.,.05,1.,.20,"call")-base
    delta_only=g.delta*move
    delta_gamma=g.delta*move+0.5*g.gamma*move**2
    rows.append({"spot_change":move,"exact":exact,"delta_only":delta_only,"delta_gamma":delta_gamma})
pd.DataFrame(rows)

,spot_change,exact,delta_only,delta_gamma
0,-10,-5.359361,-6.368307,-5.430206
1,-5,-2.939711,-3.184153,-2.949628
2,-1,-0.627365,-0.636831,-0.627450
3,1,0.646125,0.636831,0.646212
4,5,3.407323,3.184153,3.418678
5,10,7.212370,6.368307,7.306407


**What I took from it.** Delta is only a local linear approximation. Gamma improves the estimate for moderate moves, but both approximations weaken as the market moves farther from the expansion point.